# Capstone B · Predice y explica

**Formación Pública — Capa B · Ciencia de datos aplicada · Proyecto final**

Tu proyecto de cierre de la Capa B: planteas un **problema predictivo**, entrenas un modelo, lo
evalúas **con honestidad** y lo documentas con una **model card**. Integra B1–B7 (features, ML,
árboles, clasificación, pipelines, despliegue, series).

Sigue el **Ciclo Pública** con foco en *Modela*. Al final hay autoevaluación y rúbrica.

**Entregable:** este notebook completo (problema, modelo, evaluación en prueba y model card).
Súbelo a tu GitHub como pieza de portafolio.

## Paso 0 · Elige tus datos
- **Opción A (recomendada):** dataset provisto `compras_ml.csv` — compras de alimentos reales de
  ChileCompra con `cantidad`, `tamano_num`, `monto_total`, `categoria`, `tamano_proveedor`,
  `region_comprador`.
- **Opción B:** trae un CSV de tu institución con un objetivo numérico o categórico a predecir.

In [ ]:
import os, urllib.request
import pandas as pd

ARCHIVO = "compras_ml.csv"   # cámbialo por tu archivo si usas la Opción B
if not os.path.exists(ARCHIVO):
    try:
        url = "https://raw.githubusercontent.com/formacionpublica-bootcamp/bootcamp/main/B8-capstone/compras_ml.csv"
        urllib.request.urlretrieve(url, ARCHIVO)
    except Exception:
        print("Si estás en Colab, sube el archivo manualmente.")

df = pd.read_csv(ARCHIVO)
print(f"{len(df)} filas, {len(df.columns)} columnas")
df.head()

## Paso 1 · Plantea el problema predictivo
¿Qué quieres **predecir** y con qué? Define si es **regresión** (un número, ej. `monto_total`) o
**clasificación** (una categoría, ej. `tamano_proveedor`). Buen problema = útil y respondible con
estas variables.

**✍️ Mi problema:** predecir el **monto_total** de una orden (regresión) a partir de la **cantidad** de artículos y el **tamaño del proveedor** (`tamano_num`).

## Paso 2 · Features y objetivo
Separa las **features** (`X`, lo que usas para predecir) del **objetivo** (`y`, lo que predices).
Cuida no incluir en `X` información que "filtre" la respuesta (*leakage*).

In [ ]:
X = df[["cantidad", "tamano_num"]]
y = df["monto_total"]
print("X:", list(X.columns), "| y:", y.name)

## Paso 3 · Train / test (con semilla)
Reserva una parte de los datos para **evaluar con honestidad**. La semilla (`random_state`) hace el
resultado reproducible.

In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Entrenamiento: {len(X_train)} | Prueba: {len(X_test)}")

## Paso 4 · Entrena el modelo
Entrena un modelo con los datos de **entrenamiento**. Empieza simple (regresión lineal o un árbol);
lo simple e interpretable suele bastar.

In [ ]:
from sklearn.linear_model import LinearRegression
modelo = LinearRegression()
modelo.fit(X_train, y_train)
print("Modelo entrenado. Coeficientes:", dict(zip(X.columns, modelo.coef_.round(1))))

## Paso 5 · Evalúa con honestidad (en PRUEBA)
Mide el error en los datos de **prueba** (que el modelo no vio). Para regresión, el MAE (error
absoluto medio) es claro: "en promedio me equivoco en $X".

In [ ]:
from sklearn.metrics import mean_absolute_error
pred = modelo.predict(X_test)
mae = mean_absolute_error(y_test, pred)
print(f"MAE en prueba: ${mae:,.0f} CLP")

## Paso 6 · Model card y conclusión
Una *model card* documenta el modelo para que otros lo usen con responsabilidad.

- **Uso previsto:** estimar de forma orientativa el monto de una compra de alimentos según cantidad
  y tamaño de proveedor. **No** sirve para aprobar/rechazar compras ni para fiscalizar.
- **Datos:** órdenes de compra de licitaciones de ChileCompra (rubro alimentos), muestra.
- **Variables:** `cantidad`, `tamano_num`.
- **Desempeño:** MAE en prueba ≈ ver Paso 5 (orden de magnitud de cientos de miles de CLP).
- **Límites:** la relación monto≈precio×cantidad lo hace intuitivo pero simple; ignora el tipo de
  producto y precios unitarios muy distintos.
- **Riesgos éticos:** una estimación no es un juicio sobre un organismo o proveedor; usarla como
  control automático sería injusto. Requiere supervisión humana.

In [ ]:
# Autoevaluación suave — confirma que hiciste cada paso (no califica la calidad del modelo).
hechos = {
    "Definiste X e y": "X" in dir() and X is not None and "y" in dir() and y is not None,
    "Separaste prueba (X_test)": "X_test" in dir() and X_test is not None,
    "Entrenaste un modelo": "modelo" in dir() and modelo is not None,
    "Evaluaste en prueba (mae)": "mae" in dir() and mae is not None,
}
for paso, ok in hechos.items():
    print(("✅ " if ok else "⬜ ") + paso)
print("\nEvalúate con la rúbrica del README (aprobado: 12/18).")

## Rúbrica (autoevalúate, 0–3 cada una)
| Criterio | 0–3 |
|---|---|
| Problema predictivo bien planteado | |
| Features y objetivo correctos (sin *leakage*) | |
| Train/test bien hecho (con semilla) | |
| Modelo entrenado | |
| Evaluación honesta (métrica en PRUEBA) | |
| Model card (límites y ética) | |

**Aprobado: 12 de 18.** → Habilita la **certificación: Data Scientist del sector público**
(junto con el Proyecto Final, si tu institución lo pide más extenso).